1. Criar tabela queimadas_raw
2. Transformações nos dados -> criando outra tabela
3. Fazer uma análise de focos de incendio

## Criando tabela queimadas_raw

In [0]:
%sql
-- Criando a tabela usando SQL:
CREATE OR REPLACE TABLE workspace.default.queimadas_raw
USING delta -- Tabela final sera delta
AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/raw_data/queimadas.csv',
  format => 'csv',
  header => true,
  inferSchema => true,
  schemaEvolutionMode => 'none' -- desabilita a criacao da coluna _rescued_data
)


In [0]:
%sql
select * from workspace.default.queimadas_raw
limit 5;

## Lendo tabela -> Criando DF

In [0]:
database_name = 'workspace.default'
table_name = 'queimadas_raw'

df_raw = spark.read.table(f'{database_name}.{table_name}')
display(df_raw.limit(100))

In [0]:
df_raw.show(100)

## Limpeza de dados

Ajustes de null

In [0]:
from pyspark.sql.functions import col
df_raw.filter(col('dias_sem_chuva') == -999).count()

In [0]:
# Quantidade de linhas sem dados de dias de chuva por localidade (long, lat)
# -999 = null, nesse dataset

display(
    df_raw.filter(col('dias_sem_chuva') == -999)
    .groupBy('sigla_uf')
    .count()
    .orderBy(col('count'), ascending=False)
)

In [0]:
# converter para null

df_cleaned = df_raw.replace(-999, None, 'dias_sem_chuva')

df_cleaned.filter(col('dias_sem_chuva').isNull()).count()

display(df_cleaned.limit(100))

Criando pk

In [0]:
from pyspark.sql.functions import monotonically_increasing_id # Cria ID aleatorio, nao é sequencial 

df_pk = df_cleaned.withColumn('id', monotonically_increasing_id())

display(df_pk.limit(100))

## Escrevendo tabela

In [0]:
path_table = 'workspace.default'
table_name = 'queimadas_cleaned'

df_pk.write.mode('overwrite').saveAsTable(f'{path_table}.{table_name}')

In [0]:
df_result = spark.read.table(f'{path_table}.{table_name}')
display(df_result.limit(100))

## O que evitar no spark


In [0]:
.toPandas() # Remove todo o poder do paralelismo do spark (tudo pra RAM)
.collect()  # transforma em uma lista de rows (traz tudo pra RAM do driver) -> adeus paralelismo (OOM)
# Evite fazer shuffle (embaralhamento) -> Operacao muito custosa